## 🤖 Ensamble de Modelos  

---

### ¿Qué es un modelo de ensamble?  
Un **modelo de ensamble** combina múltiples modelos base (débilmente o fuertemente predictivos) para generar un modelo más robusto y preciso. En lugar de confiar en una sola predicción, se aprovecha la *sabiduría del grupo* 👥.  


###  Intuición del modelo  
Imagina que preguntas a varias personas la misma pregunta complicada 🤔.  
Aunque cada una pueda equivocarse, **al promediar sus respuestas obtienes un resultado más confiable**.  
👉 Esto mismo ocurre en los modelos de ensamble: la combinación de varios reduce errores individuales.  



### Ventajas de los modelos  
✨ Mayor precisión en predicciones  
🛡️ Menor riesgo de *overfitting* (dependiendo del método)  
🔍 Pueden capturar diferentes patrones en los datos  
⚡ Son versátiles: se aplican tanto a clasificación como regresión  


###  Ejemplos de modelos de ensamble  
- 🌲 **Bagging (Bootstrap Aggregating)**: como *Random Forest*  
- 🚀 **Boosting**: como *Gradient Boosting, AdaBoost, XGBoost, LightGBM*  
- 🧩 **Stacking**: combina distintos algoritmos (ej. árboles + regresión logística)  
- 🗳️ **Voting Classifier**: mayoría de votos entre modelos  


### 🚀 Potenciación de Gradiente (Boosting)  

---

#### ¿Qué es un modelo de ensamble?  
Un **modelo de ensamble** combina varios modelos base (llamados *weak learners*, usualmente árboles poco profundos 🌳) para formar un modelo más fuerte y preciso. En *boosting*, estos modelos se entrenan **de manera secuencial**, corrigiendo los errores del anterior.  



####  Intuición del modelo  
Imagina que un grupo de estudiantes resuelve un examen 📘.  
- El primero se equivoca en varias preguntas.  
- El segundo revisa y se enfoca en esas preguntas falladas.  
- El tercero hace lo mismo con los errores que quedan.  

👉 Al final, la combinación de todos mejora el resultado global.  
Eso es **boosting**: cada nuevo modelo se entrena para corregir los errores de los anteriores.  



####  Ventajas de los modelos  
✨ Alta precisión en comparación con modelos individuales.  
🛡️ Reduce el sesgo al enfocarse en errores previos.  
⚡ Flexible: puede usarse tanto en clasificación como regresión.  
📊 Permite ajustar la importancia de cada modelo mediante tasas de aprendizaje (*learning rate*).  



####  Ejemplos de modelos de ensamble con boosting  
- 🔥 **AdaBoost**: ajusta pesos de las observaciones mal clasificadas.  
- 🚀 **Gradient Boosting**: usa gradiente descendente para mejorar el ajuste.  
- ⚡ **XGBoost**: versión optimizada, muy rápida y eficiente.  
- 🌱 **LightGBM**: diseñado para grandes volúmenes de datos.  
- 🌀 **CatBoost**: muy efectivo con variables categóricas.  


##  Ejemplo práctico: Predicción del Precio de Autos - Rusty Bargain

---

Rusty Bargain es un servicio de coches usados que desea crear una app para estimar el valor de mercado de un vehículo.


### 🎯 Objetivos 

- Predecir el precio de un coche en euros.
- Comparar modelos diferentes: regresión lineal, árbol de decisión, bosque aleatorio y potenciación del gradiente (LightGBM).
- Implementar manualmente una regresión lineal con descenso por gradiente.
- Medir:
  - Precisión: RECM (RMSE)
  - Tiempo de entrenamiento
  - Velocidad de predicción


### Consideraciones

- La regresión lineal sirve como prueba de cordura.
- Potenciación del gradiente debe funcionar mejor que regresión lineal, si no, algo está mal.
- LightGBM y CatBoost manejan categóricas; XGBoost requiere OHE.
- Usa `%%time` para medir tiempos en Jupyter.

In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time

from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import lightgbm as lgb

In [18]:
df = pd.read_csv("https://practicum-content.s3.us-west-1.amazonaws.com/datasets/car_data.csv")

In [19]:
df.head()

,DateCrawled,Price,VehicleType,RegistrationYear,Gearbox,Power,Model,Mileage,RegistrationMonth,FuelType,Brand,NotRepaired,DateCreated,NumberOfPictures,PostalCode,LastSeen
0,24/03/2016 11:52,480,NaN,1993,manual,0,golf,150000,0,petrol,volkswagen,NaN,24/03/2016 00:00,0,70435,07/04/2016 03:16
1,24/03/2016 10:58,18300,coupe,2011,manual,190,NaN,125000,5,gasoline,audi,yes,24/03/2016 00:00,0,66954,07/04/2016 01:46
2,14/03/2016 12:52,9800,suv,2004,auto,163,grand,125000,8,gasoline,jeep,NaN,14/03/2016 00:00,0,90480,05/04/2016 12:47
3,17/03/2016 16:54,1500,small,2001,manual,75,golf,150000,6,petrol,volkswagen,no,17/03/2016 00:00,0,91074,17/03/2016 17:40
4,31/03/2016 17:25,3600,small,2008,manual,69,fabia,90000,7,gasoline,skoda,no,31/03/2016 00:00,0,60437,06/04/2016 10:17


### EDA y Limpieza de datos

En esta celda realizamos una serie de pasos clave para preparar los datos antes de entrenar nuestros modelos:

1. **Filtrado de valores extremos (outliers):**
   - `Price` se restringe entre 100 y 50,000 euros para eliminar errores o valores atípicos.
   - `RegistrationYear` se limita entre 1950 y 2022 para asegurarse de que los años sean válidos.
   - `Power` (potencia del vehículo) se restringe entre 10 y 500 caballos de fuerza para evitar valores irreales.

2. **Eliminación de columnas irrelevantes:**
   - Se eliminan columnas que no aportan valor predictivo como:
     - `NumberOfPictures` (todos los valores son 0)
     - Fechas (`DateCrawled`, `DateCreated`, `LastSeen`)
     - `PostalCode` (identificador geográfico muy granular)

3. **Eliminación de filas con valores faltantes:**
   - `df.dropna()` descarta cualquier fila que contenga `NaN`, garantizando que el modelo no se entrene con datos incompletos.

4. **Codificación de variables categóricas:**
   - Se seleccionan las columnas categóricas y se transforman en variables dummy con `pd.get_dummies()`, usando `drop_first=True` para evitar multicolinealidad.

5. **Definición de variables predictoras y objetivo:**
   - `X`: todas las columnas excepto `'Price'`, que serán las características usadas para predecir.
   - `y`: la columna `'Price'`, que es nuestro objetivo.

6. **División del conjunto de datos:**
   - Se divide en conjunto de entrenamiento (`X_train`, `y_train`) y prueba (`X_test`, `y_test`) usando un 75% para entrenar y 25% para evaluar.


In [20]:
df = df[df['Price'].between(100, 50000)]
df = df[df['RegistrationYear'].between(1950, 2022)]
df = df[df['Power'].between(10, 500)]

In [21]:
df = df.drop(columns=['NumberOfPictures', 'DateCrawled', 'DateCreated', 'LastSeen', 'PostalCode'])
df = df.dropna()

In [22]:
categorical = ['VehicleType', 'Gearbox', 'Model', 'FuelType', 'Brand', 'NotRepaired']
df = pd.get_dummies(df, columns=categorical, drop_first=True)

In [23]:
df.info(), display(df.head())

<class 'pandas.core.frame.DataFrame'>
Index: 232530 entries, 3 to 354367
Columns: 306 entries, Price to NotRepaired_yes
dtypes: bool(301), int64(5)
memory usage: 77.4 MB


,Price,RegistrationYear,Power,Mileage,RegistrationMonth,VehicleType_convertible,VehicleType_coupe,VehicleType_other,VehicleType_sedan,VehicleType_small,...,Brand_seat,Brand_skoda,Brand_smart,Brand_subaru,Brand_suzuki,Brand_toyota,Brand_trabant,Brand_volkswagen,Brand_volvo,NotRepaired_yes
3,1500,2001,75,150000,6,False,False,False,False,True,...,False,False,False,False,False,False,False,True,False,False
4,3600,2008,69,90000,7,False,False,False,False,True,...,False,True,False,False,False,False,False,False,False,False
5,650,1995,102,150000,10,False,False,False,True,False,...,False,False,False,False,False,False,False,False,False,True
6,2200,2004,109,150000,8,True,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
10,2000,2004,105,150000,12,False,False,False,True,False,...,False,False,False,False,False,False,False,False,False,False


(None, None)

In [24]:
X = df.drop('Price', axis=1)
y = df['Price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

### Baseline (Regresion lineal)

---

In [25]:
%%time
lr = LinearRegression()
lr.fit(X_train, y_train)
preds_lr = lr.predict(X_test)
rmse_lr = root_mean_squared_error(y_test, preds_lr)
print(f"RMSE Linear Regression: {rmse_lr:.2f}")

RMSE Linear Regression: 2502.57
CPU times: total: 13.4 s
Wall time: 2.27 s


### Random  forest

---

In [26]:
%%time
forest = RandomForestRegressor(n_estimators=10, max_depth=3, random_state=42)
forest.fit(X_train, y_train)
preds_forest = forest.predict(X_test)
rmse_forest = root_mean_squared_error(y_test, preds_forest)
print(f"RMSE Random Forest: {rmse_forest:.2f}")

RMSE Random Forest: 2772.93
CPU times: total: 4.67 s
Wall time: 4.81 s


### ⚡LightGBM

---

In [27]:
%%time
lgb_train = lgb.Dataset(X_train, y_train)

params = {
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.1,
    'max_depth': 10,
    'verbose': -1
}

gbm = lgb.train(params, lgb_train, num_boost_round=100)
preds_lgb = gbm.predict(X_test)

rmse_lgb = root_mean_squared_error(y_test, preds_lgb)

print(f"RMSE LightGBM: {rmse_lgb:.2f}")

RMSE LightGBM: 1650.43
CPU times: total: 6.77 s
Wall time: 788 ms


### ⚡XGBoost

---

In [28]:
%%time

# Convertir datasets al formato de XGBoost
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)

# Parámetros de entrenamiento
params = {
    'objective': 'reg:squarederror',  # regresión
    'eval_metric': 'rmse',            # métrica de evaluación
    'eta': 0.1,                       # learning rate
    'max_depth': 10,                  # profundidad máxima
    'verbosity': 0
}

# Entrenamiento del modelo
xgb_model = xgb.train(params, dtrain, num_boost_round=100)

# Predicciones
preds_xgb = xgb_model.predict(dtest)

# Calcular RMSE
rmse_xgb = root_mean_squared_error(y_test, preds_xgb)

print(f"RMSE XGBoost: {rmse_xgb:.2f}")

RMSE XGBoost: 1535.92
CPU times: total: 30.8 s
Wall time: 2.26 s


### 📊 Comparación final de modelos

In [29]:
modelos = ['LinearRegression',  'RandomForest', 'LightGBM', 'XGBoost']
rmses = [rmse_lr,  rmse_forest, rmse_lgb, rmse_xgb]
pd.DataFrame({'Modelo': modelos, 'RMSE': rmses}).sort_values(by='RMSE').round()

,Modelo,RMSE
3,XGBoost,1536.0
2,LightGBM,1650.0
0,LinearRegression,2503.0
1,RandomForest,2773.0


## Para cerrar 💬🤔

----

1. Explica en tus palabras los conceptos de:
    - Aprendizaje en ensamble
    - Potenciación del gradiente (*Boosting*) 
2. En este punto: ¿Tienes algun modelo preferido? 

## 🚀 Para seguir aprendiendo :

---

- 📚 Vuelve a revisar este notebook y trata resolver por tu cuenta el proyecto nuevamente
- 💬 Recuerda que en Discord puedes dejar todos tus comentarios y dudas sobre el contenido del sprint en [Sprint 14](https://discord.com/channels/1081207584104656986/1270074296395497513).
    - 📝 Si tienes preguntas sobre tu proyecto, usa el canal `#project` para recibir ayuda y compartir ideas.
    - 🤝 Aprovecha el espacio de `CoLearning` para aclarar tus dudas junto con otros estudiantes e instructores: [Co-Learning](https://discord.com/channels/1081207584104656986/1197953851391746119).
- 📅 ¿Necesitas ayuda personalizada? Puedes agendar una sesión `1:1` conmigo aquí: [1:1 Roman Castillo](https://scheduler.zoom.us/roman-castillo/1-1-roman-castillo).

- Por último hazme paro y responde la encuesta al final de la sesión, me sirve para poder ayudarte mejor 

¡Sigue practicando y no dudes en pedir apoyo cuando lo necesites! 💪✨